In [2]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score


In [3]:
data = pd.read_csv("resources/dataset.csv")
target_col = "event"
group_col = "user_id"   # used only for splitting, not as a feature

y = data[target_col]
groups = data[group_col]
X = data.drop(columns=[target_col])   # unused columns get dropped by the ColumnTransformer


In [6]:
X['action'].unique().shape[0]  # check number of unique actions

# show unique values
X['action'].unique()

<StringArray>
[  'lying_down',      'holding',     'reaching',      'sitting',
      'walking',     'standing',   'picking_up', 'putting_down',
        'using']
Length: 9, dtype: str

## Unnecessary Features

- user_id
- row_id
- split
- gender
- chronotype
- light_level
- action_conf
- object_conf
- lives_with_others

## Target
- event

## Numerical
- minute_of_day
- hour_sin
- hour_cos
- age
- dwell_seconds


## Categorical
- day_of_week
- is_weekend
- is_school_holiday
- occupation
- object
- action
- prev_action
- object_in_hand
- zone
- is_off_routine


In [1]:
num_scale = ["age", "minute_of_day"]
num_log   = ["dwell_seconds"]
num_pass  = ["hour_sin", "hour_cos"]
binary    = ["is_weekend", "is_school_holiday", "is_off_routine"]
cat       = ["day_of_week", "occupation", "object", "action",
             "prev_action", "object_in_hand", "zone"]


In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
pre = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), num_scale),
        ("log", Pipeline([
            ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("sc", StandardScaler()),
        ]), num_log),
        ("pass", "passthrough", num_pass + binary),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    ],
    remainder="drop", 
)


In [11]:
pre = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), num_scale),
        ("log", Pipeline([
            ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("sc", StandardScaler()),
        ]), num_log),
        ("pass", "passthrough", num_pass + binary),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    ],
    remainder="drop", 
)


In [12]:
full_pipeline = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_jobs=-1, random_state=42, class_weight="balanced")),
])

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

In [13]:
param_grid = {
    "clf__n_estimators": [50, 100, 200, 300],
    "clf__max_depth": [10, 20, 30],
}
search = GridSearchCV(
    full_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=GroupKFold(n_splits=5),
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train, groups=groups_train)

print("Best params:", search.best_params_)
print("Best CV F1:", search.best_score_)


Fitting 5 folds for each of 12 candidates, totalling 60 fits


KeyboardInterrupt: 

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

# ---------- Feature importances ----------
names = best_model.named_steps["pre"].get_feature_names_out()
imp = pd.Series(best_model.named_steps["clf"].feature_importances_, index=names)
print(imp.sort_values(ascending=False).head(15))